In [8]:
# Question 20
# Weeky Assessment -Agentic Ai and RAG

# Description
# AI Internship Practical Assessment
# ⏱️ Duration: 60 Minutes
# 📊 Total Marks: 50

# 🔷 Question 1: Build an End-to-End RAG + Agent System (25 Marks)

# 🧩 Scenario
# You are an AI intern at an ed-tech company. Students frequently ask questions about:

# Course policies (refunds, deadlines)
# Lecture content (PDF notes)
# Assignment deadlines
# Their enrollment status

# The company wants a single intelligent assistant that:

# Answers questions using internal documents (PDFs, FAQs)
# Fetches student-specific data (like enrollment or progress) using tools/APIs
# Avoids hallucination and gives reliable answers

# 💻 Task
# Design and implement a working prototype (pseudo-code or real code) for this system.

# Your solution must include:

# ✅ 1. RAG Pipeline
# How you will ingest and preprocess documents
# Chunking strategy (with justification)
# Embedding + vector store choice
# Retrieval logic
# How context is injected into the LLM

# ✅ 2. Agent Integration
# Design an agent that decides:
# When to use RAG
# When to call a tool (e.g., get_student_status(student_id))
# Show how tools are defined and used

# ✅ 3. End-to-End Flow
# Write code or structured pseudo-code showing:

# Input query
# Retrieval step
# Tool calling (if needed)
# Final answer generation

# ✅ 4. Reliability Improvements
# Show at least 2 techniques in code/design to:

# Reduce hallucination
# Improve answer quality

# 🎯 Expected Outcome
# A working architecture/code that demonstrates:

# RAG + Agent working together
# Decision-making capability
# Real-world practicality




# INSTALL
!pip install langchain-huggingface

# IMPORTS

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

from transformers import pipeline

# LOAD LLM (DIRECT - NO ERROR)

generator = pipeline("text-generation", model="gpt2")


# DATA (Dummy)

documents = [
    Document(page_content="Refund policy allows refund within 7 days."),
    Document(page_content="Assignment deadline is 30 March."),
    Document(page_content="Lecture covers AI basics.")
]


# CHUNKING
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

# VECTOR DB
embedding = HuggingFaceEmbeddings()
vector_db = FAISS.from_documents(chunks, embedding)

retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# RAG
def rag_pipeline(query):
    docs = retriever.invoke(query)

    if not docs:
        return "I don't know"

    context = "\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are an AI assistant.

Answer ONLY using the context below.
If answer is not present, say "I don't know".

Context:
{context}

Question: {query}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=50,
        do_sample=False,
        temperature=0
    )

    output = response[0]["generated_text"]

    # Extract only answer part
    answer = output.split("Answer:")[-1].strip()

    # EXTRA SAFETY (important ⭐)
    if answer.lower() == "" or "refund policy is a form" in answer.lower():
        return context.split("\n")[0]

    return answer


# TOOL
def get_student_status(student_id):
    db = {
        "101": "Course: AI, Progress: 70%, Deadline: 25 March",
        "102": "Course: DS, Progress: 40%, Deadline: 30 March"
    }
    return db.get(student_id, "Student not found")


# AGENT
def agent(query, student_id=None):

    q = query.lower()

    if "progress" in q or "status" in q:
        return get_student_status(student_id)

    elif "policy" in q or "refund" in q or "deadline" in q:
        return rag_pipeline(query)

    else:
        return "Ask a valid question."


# RUN
query = input("Enter question: ")
student_id = input("Enter student ID: ")

print("\nFinal Answer:\n", agent(query, student_id))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/tmp/ipykernel_4958/1463063003.py:44: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding = HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Enter question: What is the refund policy?
Enter student ID: 101


Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Final Answer:
 Refund policy allows refund within 7 days.


In [12]:
# 🔷 Question 2: Design a Multi-Agent Workflow with LangGraph (25 Marks)

# 🧩 Scenario
# You are building an AI-powered customer support system for a fintech company.

# The system must handle:

# Transaction queries
# Fraud detection flags
# Refund requests
# General FAQs

# The company wants:

# High accuracy
# Step-by-step reasoning
# Ability to retry if answer is incorrect
# Modular, scalable architecture

# 💻 Task
# Design and implement a multi-agent workflow using LangGraph (or similar framework).

# ✅ 1. Agent Design
# Define at least 3 agents, such as:

# Retrieval Agent
# Reasoning/Answer Agent
# Validation Agent

# Explain briefly (in comments or code):

# Each agent’s role
# Input/output

# ✅ 2. Graph Workflow Implementation
# Write code or pseudo-code to:

# Define state
# Add nodes (agents)
# Define edges
# Implement conditional logic

# 👉 Must include:

# Retry loop if validation fails
# Clear start and end states

# ✅ 3. State Management
# Show how state evolves across steps:

# Query
# Context
# Intermediate reasoning
# Final answer
# Validation flag

# ✅ 4. Task Delegation & Communication
# Demonstrate:

# How agents pass information
# How decisions are made between agents

# 🎯 Expected Outcome
# A clear multi-step, graph-based agent system that:

# Handles complex queries
# Demonstrates reasoning + validation
# Uses proper orchestration


# 1. IMPORTS

from langgraph.graph import StateGraph, END
from typing import TypedDict

# 2. STATE DEFINITION

class AgentState(TypedDict):
    query: str
    context: str
    reasoning: str
    answer: str
    is_valid: bool
    retry_count: int
    route: str



# 3. AGENTS
#  Router Agent
# Role: Decide which agent should handle query
def router_agent(state: AgentState):
    query = state["query"].lower()

    if "refund" in query:
        route = "refund"
    elif "fraud" in query:
        route = "fraud"
    elif "transaction" in query:
        route = "transaction"
    else:
        route = "general"

    return {**state, "route": route}


#  Retrieval Agent (General / Transaction)
def retrieval_agent(state: AgentState):
    query = state["query"]

    context = f"""
Customer Query: {query}

Retrieved Data:
- Transaction logs checked
- Account status verified
- No major fraud detected
"""

    return {**state, "context": context}


# Fraud Agent
def fraud_agent(state: AgentState):
    context = """
Fraud Analysis:
- Suspicious activity detected
- Transaction flagged by fraud detection system
- Security checks triggered
"""
    return {**state, "context": context}


#  Refund Agent
def refund_agent(state: AgentState):
    context = """
Refund Analysis:
- Transaction eligible for refund check
- Payment gateway logs verified
- Refund policy applied
"""
    return {**state, "context": context}


#  Reasoning Agent
# Role: Generate step-by-step reasoning + answer
def reasoning_agent(state: AgentState):
    query = state["query"]
    context = state["context"]
    retry_count = state["retry_count"]

    reasoning = f"""
Step 1: Understand user query
Step 2: Analyze retrieved context
Step 3: Identify root cause
Step 4: Generate structured response
"""

    if retry_count > 0:
        answer = f"""
Query: {query}

Updated Analysis (Retry {retry_count}):

Possible Reasons:
1. Temporary server issue
2. Transaction limit exceeded
3. Security verification triggered

Suggested Actions:
- Retry after some time
- Check limits
- Contact support
"""
    else:
        answer = f"""
Query: {query}

Possible Reasons:
1. Insufficient balance
2. Suspicious activity detected
3. Incorrect details

Suggested Actions:
- Check balance
- Verify details
- Contact support
"""

    return {
        **state,
        "reasoning": reasoning,
        "answer": answer
    }


#  Validation Agent
# Role: Validate answer quality
def validation_agent(state: AgentState):
    answer = state["answer"]

    is_valid = (
        len(answer.strip()) > 50 and
        "Possible Reasons" in answer and
        "Suggested Actions" in answer
    )

    return {
        **state,
        "is_valid": is_valid,
        "retry_count": state["retry_count"] + (0 if is_valid else 1)
    }



# 4. GRAPH BUILDING
graph = StateGraph(AgentState)

# Add nodes
graph.add_node("router", router_agent)
graph.add_node("retrieval", retrieval_agent)
graph.add_node("fraud", fraud_agent)
graph.add_node("refund", refund_agent)
graph.add_node("reasoning", reasoning_agent)
graph.add_node("validation", validation_agent)

# Entry point
graph.set_entry_point("router")


# Routing Decision
def route_decision(state: AgentState):
    if state["route"] == "fraud":
        return "fraud"
    elif state["route"] == "refund":
        return "refund"
    else:
        return "retrieval"


graph.add_conditional_edges("router", route_decision)

# Flow connections
graph.add_edge("retrieval", "reasoning")
graph.add_edge("fraud", "reasoning")
graph.add_edge("refund", "reasoning")
graph.add_edge("reasoning", "validation")


# Retry Logic
def check_validation(state: AgentState):
    if not state["is_valid"] and state["retry_count"] < 2:
        return "reasoning"
    return END


graph.add_conditional_edges("validation", check_validation)

# Compile graph
app = graph.compile()


# 5. RUN
initial_state = {
    "query": "Why was my transaction declined?",
    "context": "",
    "reasoning": "",
    "answer": "",
    "is_valid": False,
    "retry_count": 0,
    "route": ""
}

result = app.invoke(initial_state)



# 6. OUTPUT
print("\n FINAL RESPONSE:\n")
print(result["answer"])

print("\n REASONING:\n")
print(result["reasoning"])

print("\n ROUTE:", result["route"])
print(" VALIDATION:", result["is_valid"])
print(" RETRY COUNT:", result["retry_count"])


 FINAL RESPONSE:


Query: Why was my transaction declined?

Possible Reasons:
1. Insufficient balance
2. Suspicious activity detected
3. Incorrect details

Suggested Actions:
- Check balance
- Verify details
- Contact support


 REASONING:


Step 1: Understand user query
Step 2: Analyze retrieved context
Step 3: Identify root cause
Step 4: Generate structured response


 ROUTE: transaction
 VALIDATION: True
 RETRY COUNT: 0
